In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from datasets import load_dataset

ds_reduced = load_dataset(
    "supermarine45/4be-dataset",
    data_files={
        "train_reduced": [
            "Option1/option1_nf_unsw_dos_as_ddos_reduced_schema/attack/train/*.csv",
            "Option1/option1_nf_unsw_dos_as_ddos_reduced_schema/normal/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos_reduced_schema/attack/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos_reduced_schema/normal/train/*.csv"
        ]
    }
)

ds_full = load_dataset(
    "supermarine45/4be-dataset",
    data_files={
        "train_full": [
            "Option1/option1_nf_unsw_dos_as_ddos/attack/train/*.csv",
            "Option1/option1_nf_unsw_dos_as_ddos/normal/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos/attack/train/*.csv",
            "Option2/option2_nf_unsw_base_cse_native_ddos/normal/train/*.csv"
        ]
    }
)

In [ ]:
ds_full["train_full"][1000]

In [ ]:
print("df_reduced columns:")
print(df_reduced.columns.tolist())
print("\ndf_full columns:")
print(df_full.columns.tolist())

In [ ]:
# Convert to pandas dataframes
df_reduced = ds_reduced["train_reduced"].to_pandas()
df_full = ds_full["train_full"].to_pandas()

print(f"df_reduced shape: {df_reduced.shape}")
print(f"df_full shape: {df_full.shape}")

In [ ]:
# COLUMN Statistics

# Analyze Attack column distribution
attack_dist = df_full['Attack'].value_counts(normalize=True) * 100
attack_counts = df_full['Attack'].value_counts()

print('=' * 80)
print('ATTACK COLUMN DISTRIBUTION - PERCENTAGE AND COUNTS')
print('=' * 80)
print('\nPercentage breakdown:')
print(attack_dist.sort_values(ascending=False))

print('\nAbsolute counts:')
print(attack_counts.sort_values(ascending=False))

print('\nSummary:')
benign_pct = (df_full['Attack'] == 'Benign').sum() / len(df_full) * 100
attack_pct = 100 - benign_pct
print(f'Benign: {benign_pct:.2f}% ({(df_full["Attack"] == "Benign").sum():,} rows)')
print(f'DDoS/Attack: {attack_pct:.2f}% ({(df_full["Attack"] != "Benign").sum():,} rows)')
print(f'Total rows: {len(df_full):,}')

In [ ]:
print('=' * 80)
print('COMPREHENSIVE DATA STATISTICS')
print('=' * 80)

# 1. DATASET SHAPE AND SIZE
print('\n1. DATASET SHAPE AND SIZE')
print('-' * 80)
print(f'Total rows: {len(df_reduced):,}')
print(f'Total columns: {len(df_reduced.columns):,}')
print(f'Memory usage: {df_reduced.memory_usage(deep=True).sum() / 1024**3:.2f} GB')
print(f'Approx rows per MB: {len(df_reduced) / (df_reduced.memory_usage(deep=True).sum() / 1024**2):.0f}')

# 2. MISSING VALUES
print('\n2. MISSING VALUES')
print('-' * 80)
missing = df_reduced.isnull().sum()
missing_pct = (missing / len(df_reduced) * 100).round(2)
missing_summary = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
if len(missing_summary) > 0:
    print(missing_summary)
else:
    print('No missing values found!')

# 3. DATA TYPES
print('\n3. DATA TYPES DISTRIBUTION')
print('-' * 80)
dtype_counts = df_reduced.dtypes.value_counts()
print(dtype_counts)

# 4. NUMERIC COLUMN STATISTICS
print('\n4. NUMERIC COLUMNS - SUMMARY STATISTICS')
print('-' * 80)
numeric_stats = df_reduced.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
print(numeric_stats[['min', '25%', '50%', '75%', '95%', '99%', 'max', 'std']].round(2))

# 5. ATTACK DISTRIBUTION BY SCENARIO
print('\n5. ATTACK DISTRIBUTION BY SCENARIO')
print('-' * 80)
scenario_attack = pd.crosstab(df_reduced['scenario'], df_reduced['Attack'], margins=True)
print(scenario_attack)

# 6. ATTACK DISTRIBUTION BY DATASET
print('\n6. ATTACK DISTRIBUTION BY DATASET ID')
print('-' * 80)
dataset_attack = pd.crosstab(df_reduced['dataset_id'], df_reduced['Attack'], margins=True)
print(dataset_attack)

# 7. SOURCE IP STATISTICS
print('\n7. SOURCE IP STATISTICS')
print('-' * 80)
n_unique_src_ips = df_reduced['src_ip'].nunique()
src_ip_flow_counts = df_reduced['src_ip'].value_counts()
print(f'Unique source IPs: {n_unique_src_ips:,}')
print(f'Mean flows per source: {src_ip_flow_counts.mean():.1f}')
print(f'Median flows per source: {src_ip_flow_counts.median():.1f}')
print(f'Max flows from single source: {src_ip_flow_counts.max():,}')
print(f'Min flows per source: {src_ip_flow_counts.min()}')
print(f'95th percentile flows: {src_ip_flow_counts.quantile(0.95):.0f}')
print('\nTop 10 most active source IPs:')
print(src_ip_flow_counts.head(10))

# 8. DESTINATION IP/PORT STATISTICS
print('\n8. DESTINATION IP AND PORT STATISTICS')
print('-' * 80)
n_unique_dst_ips = df_reduced['dst_ip'].nunique()
n_unique_dst_ports = df_reduced['dst_port'].nunique()
print(f'Unique destination IPs: {n_unique_dst_ips:,}')
print(f'Unique destination ports: {n_unique_dst_ports:,}')
print(f'Port range: {df_reduced["dst_port"].min()} - {df_reduced["dst_port"].max()}')
print('\nTop 10 most targeted ports:')
print(df_reduced['dst_port'].value_counts().head(10))

# 9. PROTOCOL DISTRIBUTION
print('\n9. PROTOCOL DISTRIBUTION')
print('-' * 80)
proto_counts = df_reduced['protocol'].value_counts()
proto_pct = (proto_counts / len(df_reduced) * 100).round(2)
proto_summary = pd.DataFrame({'Count': proto_counts, 'Percentage': proto_pct})
print(proto_summary)

# 10. ATTACK PATTERN BY PROTOCOL
print('\n10. ATTACK PATTERN BY PROTOCOL')
print('-' * 80)
protocol_attack = pd.crosstab(df_reduced['protocol'], df_reduced['Attack'], margins=True)
print(protocol_attack)

# 11. FLOW DURATION STATISTICS
print('\n11. FLOW DURATION STATISTICS')
print('-' * 80)
print(f'Mean duration: {df_reduced["duration"].mean():.4f} sec')
print(f'Median duration: {df_reduced["duration"].median():.4f} sec')
print(f'Min duration: {df_reduced["duration"].min():.4f} sec')
print(f'Max duration: {df_reduced["duration"].max():.4f} sec')
print(f'Std dev: {df_reduced["duration"].std():.4f} sec')

# 12. TRAFFIC RATE STATISTICS (packets_per_second, bytes_per_second)
print('\n12. TRAFFIC RATE STATISTICS')
print('-' * 80)
print('\nPackets per second:')
print(f'  Mean: {df_reduced["packets_per_second"].mean():.2f}')
print(f'  Median: {df_reduced["packets_per_second"].median():.2f}')
print(f'  95th percentile: {df_reduced["packets_per_second"].quantile(0.95):.2f}')
print(f'  99th percentile: {df_reduced["packets_per_second"].quantile(0.99):.2f}')
print(f'  Max: {df_reduced["packets_per_second"].max():.2f}')

print('\nBytes per second:')
print(f'  Mean: {df_reduced["bytes_per_second"].mean():.2f}')
print(f'  Median: {df_reduced["bytes_per_second"].median():.2f}')
print(f'  95th percentile: {df_reduced["bytes_per_second"].quantile(0.95):.2f}')
print(f'  99th percentile: {df_reduced["bytes_per_second"].quantile(0.99):.2f}')
print(f'  Max: {df_reduced["bytes_per_second"].max():.2f}')

# 13. TRAFFIC VOLUME STATISTICS
print('\n13. TRAFFIC VOLUME STATISTICS')
print('-' * 80)
print(f'Mean total packets per flow: {df_reduced["total_packets"].mean():.2f}')
print(f'Median total packets per flow: {df_reduced["total_packets"].median():.2f}')
print(f'Max total packets in single flow: {df_reduced["total_packets"].max():,}')
print(f'Mean total bytes per flow: {df_reduced["total_bytes"].mean():.2f}')
print(f'Median total bytes per flow: {df_reduced["total_bytes"].median():.2f}')
print(f'Max total bytes in single flow: {df_reduced["total_bytes"].max():,}')

# 14. PACKET SIZE ANALYSIS
print('\n14. PACKET SIZE ANALYSIS')
print('-' * 80)
print(f'Mean packet size: {df_reduced["packet_size_avg"].mean():.2f} bytes')
print(f'Median packet size: {df_reduced["packet_size_avg"].median():.2f} bytes')
print(f'Min packet size: {df_reduced["packet_size_avg"].min():.2f} bytes')
print(f'Max packet size: {df_reduced["packet_size_avg"].max():.2f} bytes')
print(f'Std dev packet size: {df_reduced["packet_size_std"].mean():.2f} bytes')

# 15. OUTBOUND BYTE RATIO ANALYSIS
print('\n15. OUTBOUND BYTE RATIO ANALYSIS (Asymmetry)')
print('-' * 80)
print(f'Mean outbound ratio: {df_reduced["outbound_byte_ratio"].mean():.4f}')
print(f'Median outbound ratio: {df_reduced["outbound_byte_ratio"].median():.4f}')
print(f'Min ratio: {df_reduced["outbound_byte_ratio"].min():.4f}')
print(f'Max ratio: {df_reduced["outbound_byte_ratio"].max():.4f}')
print(f'Flows with ratio < 0.1 (inbound heavy): {(df_reduced["outbound_byte_ratio"] < 0.1).sum():,} ({(df_reduced["outbound_byte_ratio"] < 0.1).mean()*100:.2f}%)')

# 16. ATTACK vs BENIGN COMPARISON
print('\n16. ATTACK vs BENIGN - KEY METRICS COMPARISON')
print('-' * 80)
attack_benign_compare = df_reduced.groupby('Attack')[['packets_per_second', 'bytes_per_second', 'duration', 'total_packets', 'total_bytes', 'outbound_byte_ratio']].agg(['mean', 'median', 'std', 'min', 'max'])
print(attack_benign_compare.round(2))

# 17. BURSTS AND SEEDING INFO
print('\n17. BURST AND SEEDING INFORMATION')
print('-' * 80)
print(f'Flows with seeded DDoS flag: {df_reduced["is_seeded_ddos"].sum():,} ({df_reduced["is_seeded_ddos"].mean()*100:.2f}%)')
print(f'Unique burst IDs: {df_reduced["burst_id"].nunique():,}')
print(f'Burst phases: {df_reduced["burst_phase"].nunique() if "burst_phase" in df_reduced.columns else "N/A"}')

# 18. DATASET SPLITS
print('\n18. TRAIN/TEST SPLIT')
print('-' * 80)
split_dist = df_reduced['split'].value_counts()
split_pct = (split_dist / len(df_reduced) * 100).round(2)
split_summary = pd.DataFrame({'Count': split_dist, 'Percentage': split_pct})
print(split_summary)

In [ ]:
# Drop columns with missing values
df_reduced_clean = df_reduced.drop(columns=['burst_id', 'burst_phase'], errors='ignore')

In [ ]:
# Feature policy for src_ip-window aggregated model
# Never use audit/leakage fields as model inputs.
audit_fields = [
    'Attack', 'Label', 'scenario', 'split', 'dataset_id',
    'burst_id', 'burst_phase', 'is_seeded_ddos', 'source_dataset', 'row_in_window'
]

# src_ip is the grouping key for aggregation, not a model input.
identity_fields = ['src_ip', 'dst_ip']

# Requested feature family mapped to available reduced-schema columns.
selected_feature_candidates = [
    'dst_port',                    # destination port behavior proxy
    'protocol',                    # protocol behavior
    'packets_per_second',          # packet rate
    'bytes_per_second',            # byte rate
    'duration',                    # duration
    'total_packets',               # traffic volume
    'total_bytes',                 # traffic volume
    'packet_size_avg',             # packet-size stats
    'packet_size_std',             # packet-size stats
    'outbound_byte_ratio',         # outbound ratio
    'inter_packet_arrival_mean',   # temporal behavior
    'inter_packet_arrival_std'     # temporal behavior
]

available_selected_features = [
    col for col in selected_feature_candidates if col in df_reduced.columns
]
missing_selected_features = [
    col for col in selected_feature_candidates if col not in df_reduced.columns
]

print('Selected features found:', available_selected_features)
if missing_selected_features:
    print('Selected features missing in df_reduced:', missing_selected_features)

# Build modeling frame from selected features only.
X_selected = df_reduced[available_selected_features].copy()

# Encode categorical selected features only (protocol is categorical behaviorally).
categorical_selected = [col for col in ['protocol'] if col in X_selected.columns]
if categorical_selected:
    X_selected = pd.get_dummies(X_selected, columns=categorical_selected, drop_first=True, dtype=float)

# Ensure numeric matrix and clean invalid rows.
X_selected = X_selected.apply(pd.to_numeric, errors='coerce')
X_selected = X_selected.astype(float)

# Binary target: Attack vs Benign
y_selected = (df_reduced['Attack'].astype(str).str.lower() != 'benign').astype(int)

mask_clean = ~(
    X_selected.isna().any(axis=1)
    | np.isinf(X_selected).any(axis=1)
    | y_selected.isna()
)

X_clean = X_selected.loc[mask_clean].reset_index(drop=True)
y_clean = y_selected.loc[mask_clean].reset_index(drop=True)

print(f'X_clean shape: {X_clean.shape}')
print('Class balance (0=Benign, 1=Attack):')
print(y_clean.value_counts(normalize=True).sort_index())

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF on selected feature set only
sample_size = min(10000, len(X_clean))
X_sample = X_clean.sample(n=sample_size, random_state=42)
X_vif = sm.add_constant(X_sample, has_constant='add')

print('Calculating VIF on selected features...')

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

vif_data = (
    vif_data[vif_data['Feature'] != 'const']
    .sort_values(by='VIF', ascending=False)
    .reset_index(drop=True)
)

print('\n' + '=' * 80)
print('VIF ON SELECTED FEATURES')
print('=' * 80)
print(vif_data.head(20))

high_vif_features = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
good_features = vif_data[vif_data['VIF'] <= 10]['Feature'].tolist()

print(f'\nHigh VIF features (>10): {len(high_vif_features)}')
print(high_vif_features[:20])
print(f'\nGood features (<=10): {len(good_features)}')
print(good_features[:20])

In [ ]:
import statsmodels.api as sm

# OLS on selected feature set only
X_ols = sm.add_constant(X_clean, has_constant='add')
ols_selected = sm.OLS(y_clean, X_ols).fit()

print('\n' + '=' * 80)
print('OLS RESULTS - SELECTED FEATURE SET')
print('=' * 80)
print(ols_selected.summary())

In [ ]:
# Significant features from OLS using selected feature set
print('\n' + '=' * 80)
print('SIGNIFICANT FEATURES (p < 0.05) - SELECTED SET')
print('=' * 80)

sig_features_selected = []
for feature in ols_selected.params.index:
    if feature == 'const':
        continue
    pvalue = ols_selected.pvalues[feature]
    if pvalue < 0.05:
        coef = ols_selected.params[feature]
        sig_features_selected.append({
            'feature': feature,
            'coefficient': coef,
            'p_value': pvalue,
            'abs_coef': abs(coef)
        })

sig_features_selected.sort(key=lambda x: x['abs_coef'], reverse=True)
print(f'Found {len(sig_features_selected)} significant features')
for i, feat in enumerate(sig_features_selected[:20], 1):
    print(f"{i:2d}. {feat['feature']:35s} | coef: {feat['coefficient']:12.6f} | p-value: {feat['p_value']:.2e}")

# Cross-check OLS significance with multicollinearity filter
sig_feature_names = {x['feature'] for x in sig_features_selected}
vif_ok_set = set(good_features)
final_candidate_features = sorted(sig_feature_names.intersection(vif_ok_set))

print('\n' + '=' * 80)
print('FINAL CANDIDATE FEATURES (OLS significant AND VIF <= 10)')
print('=' * 80)
print(final_candidate_features)
print(f'Total final candidate features: {len(final_candidate_features)}')

print('\nDropped by policy (audit/leakage fields):')
print(audit_fields + identity_fields)

In [ ]:
print('final_candidate_features:', final_candidate_features)
print('num_final_features:', len(final_candidate_features))

In [ ]:
def engineer_ddos_features(df):
    # Define thresholds based on common DDoS characteristics
    SMALL_PACKET_THRESHOLD = 128
    HIGH_PPS_THRESHOLD = 1000  # Based on your OLS significance of pps
    LOW_OUTBOUND_THRESHOLD = 0.1 # Based on your OLS significance of outbound ratio

    # 1. Base Aggregations
    # We use a dictionary for standard mean/max/sum operations
    agg_dict = {
        'dst_ip': 'nunique',
        'dst_port': 'nunique',
        'protocol': 'nunique',
        'packets_per_second': ['mean', 'max'],
        'bytes_per_second': ['mean', 'max'],
        'duration': ['mean', 'max'],
        'total_packets': 'sum',
        'total_bytes': 'sum',
        'packet_size_avg': ['mean', 'std'],
        'outbound_byte_ratio': ['mean', 'min'],
        'Label': 'max' # If any flow in the window is an attack, the aggregate is 1
    }

    # Group by Source IP
    # Note: If your window spans multiple bursts, you might group by ['src_ip', 'dataset_id']
    grouped = df.groupby('src_ip')
    
    # Execute standard aggregations
    features = grouped.agg(agg_dict)
    
    # Flatten MultiIndex columns (e.g., ('duration', 'mean') -> 'duration_mean')
    features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in features.columns]
    
    # 2. Custom "Share" and "Concentration" Features
    # Concentration: Ratio of flows going to the most frequent destination
    features['concentration_dst_ip'] = grouped['dst_ip'].apply(
        lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0
    )
    features['concentration_dst_port'] = grouped['dst_port'].apply(
        lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0
    )

    # Protocol Shares (TCP=6, UDP=17, ICMP=1)
    features['share_tcp'] = grouped['protocol'].apply(lambda x: (x == 6).mean())
    features['share_udp'] = grouped['protocol'].apply(lambda x: (x == 17).mean())
    features['share_icmp'] = grouped['protocol'].apply(lambda x: (x == 1).mean())

    # Behavioral Shares
    features['share_small_packets'] = grouped['packet_size_avg'].apply(
        lambda x: (x < SMALL_PACKET_THRESHOLD).mean()
    )
    features['share_high_pps'] = grouped['packets_per_second'].apply(
        lambda x: (x > HIGH_PPS_THRESHOLD).mean()
    )
    features['share_low_outbound'] = grouped['outbound_byte_ratio'].apply(
        lambda x: (x < LOW_OUTBOUND_THRESHOLD).mean()
    )
    
    # Number of flows generated by source
    features['num_flows'] = grouped.size()

    # Final cleanup: Replace NaNs from std() calculations with 0
    return features.fillna(0).reset_index()

# Integration into your notebook:
df_engineered = engineer_ddos_features(df_reduced)
df_engineered.head()

Final features

1. bytes_per_second: Measures the volume of data flow over time.

2. dst_port: Acts as a proxy for destination behavior (e.g., targeting specific services).

3. duration: The length of the network flow.

4. outbound_byte_ratio: A critical indicator of asymmetry, which is highly significant in DDoS detection.

5. packets_per_second: Measures the intensity/rate of the packet transmission.

6. aggregated features (mean, max, and sum)

7. concentration_dst_ip, concentration_dst_port

8. TCP, UDP, or ICMP\

9. share_small_packets: Identifying "noisy" small-packet floods.

10. share_low_outbound

Dropped features:

1. High Multicollinearity (Dropped due to VIF > 10): 
total_bytes, total_packets, packet_size_avg, packet_size_std, and inter_packet_arrival 

2. Audit Fields: Label, scenario, split, dataset_id, burst_id, burst_phase, and is_seeded_ddos

## Lag and Rolling Window Features
The data is not a classic evenly sampled time series, but it does contain flow order within each `src_ip` and dataset window. That makes it suitable for lag and rolling-history features that capture short-term bursts before a DDoS label appears.

In [ ]:
import numpy as np

def build_windowed_behavior_with_lags(df):
    # ---------------------------------------------------------
    # STEP 1: AGGREGATE INTO WINDOWS (Fixing Cell 63)
    # ---------------------------------------------------------
    SMALL_PACKET_THRESHOLD = 128
    HIGH_PPS_THRESHOLD = 1000
    LOW_OUTBOUND_THRESHOLD = 0.1

    agg_dict = {
        'dst_ip': 'nunique',
        'dst_port': 'nunique',
        'protocol': 'nunique',
        'packets_per_second': ['mean', 'max'],
        'bytes_per_second': ['mean', 'max'],
        'duration': ['mean', 'max'],
        'total_packets': 'sum',
        'total_bytes': 'sum',
        'packet_size_avg': ['mean', 'std'],
        'outbound_byte_ratio': ['mean', 'min'],
        'Label': 'max'
    }

    # CRITICAL FIX: Group by both IP and Window to preserve time
    grouped = df.groupby(['src_ip', 'dataset_id'])
    
    features = grouped.agg(agg_dict)
    features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in features.columns]
    
    features['concentration_dst_ip'] = grouped['dst_ip'].apply(lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0)
    features['concentration_dst_port'] = grouped['dst_port'].apply(lambda x: x.value_counts().iloc[0] / len(x) if not x.empty else 0)
    features['share_tcp'] = grouped['protocol'].apply(lambda x: (x == 6).mean())
    features['share_udp'] = grouped['protocol'].apply(lambda x: (x == 17).mean())
    features['share_icmp'] = grouped['protocol'].apply(lambda x: (x == 1).mean())
    features['share_small_packets'] = grouped['packet_size_avg'].apply(lambda x: (x < SMALL_PACKET_THRESHOLD).mean())
    features['share_high_pps'] = grouped['packets_per_second'].apply(lambda x: (x > HIGH_PPS_THRESHOLD).mean())
    features['share_low_outbound'] = grouped['outbound_byte_ratio'].apply(lambda x: (x < LOW_OUTBOUND_THRESHOLD).mean())
    features['num_flows'] = grouped.size()

    df_agg = features.fillna(0).reset_index()

    # ---------------------------------------------------------
    # STEP 2: APPLY LAGS TO WINDOWS (Fixing Cell 70)
    # ---------------------------------------------------------
    def add_window_lags(group):
        # Sort chronologically by dataset_id (window sequence)
        group = group.sort_values('dataset_id').copy()
        
        # Apply lag to the AGGREGATED features that matter most
        lag_cols = [
            'packets_per_second_mean', 
            'bytes_per_second_mean', 
            'outbound_byte_ratio_mean',
            'total_bytes_sum'
        ]
        
        for col in lag_cols:
            lag_1 = group[col].shift(1)
            lag_3 = group[col].shift(1).rolling(window=3, min_periods=1)
            
            group[f'{col}_lag_1'] = lag_1
            group[f'{col}_lag_3_mean'] = lag_3.mean()
            group[f'{col}_delta_1'] = group[col] - lag_1
            
            # Re-implementing your spike logic on windowed data
            if 'packets_per_second_mean' in col or 'bytes_per_second_mean' in col:
                group[f'{col}_spike_3'] = (group[col] > lag_3.mean() * 1.5).astype(float)
                
        return group

    # Apply lags per source IP on the windowed data
    df_final = (
        df_agg
        .groupby('src_ip', group_keys=False)
        .apply(add_window_lags)
        .reset_index(drop=True)
    )
    
    return df_final.fillna(0) # Fill initial window NaNs with 0

# Execute
df_model_ready = build_windowed_behavior_with_lags(df_reduced)
display(df_model_ready)